[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansensio/axr/blob/master/axr/00_intro.ipynb)

# Introducción al Aprendizaje por Refuerzo

> **Cuadernillo anotado para estudiantes nuevos en Machine Learning**

---

## ¿Cómo usar este cuadernillo?

Este cuadernillo está pensado para que lo leas **de arriba a abajo, sin saltarte nada**. Cada sección de texto explica el *por qué* antes de mostrar el código. Encontrarás:

- 📖 **Secciones de teoría** — conceptos explicados con analogías cotidianas.
- 💡 **Cajas de punto clave** — las ideas más importantes resumidas.
- 🔍 **Análisis línea a línea** — cada parte del código desglosada.
- ✏️ **Ejercicios de reflexión** — preguntas para verificar tu comprensión.

**No hace falta memorizar nada.** El objetivo es que al final del cuadernillo entiendas la lógica general del aprendizaje por refuerzo y puedas modificar el código con confianza.

---
## 0. Contexto: ¿dónde encaja el axr dentro del Machine Learning?

Antes de hablar de aprendizaje por refuerzo, conviene entender el panorama general. El **Machine Learning (ML)** es la rama de la inteligencia artificial que estudia cómo hacer que las computadoras aprendan patrones a partir de datos, en lugar de programarlas con reglas explícitas.

Existen tres grandes paradigmas:

```
┌─────────────────────────────────────────────────────────────────┐
│                      MACHINE LEARNING                           │
│                                                                  │
│  ┌─────────────────────┐  ┌──────────────────┐  ┌───────────┐  │
│  │  APRENDIZAJE        │  │  APRENDIZAJE     │  │ APRENDIZ. │  │
│  │  SUPERVISADO        │  │  NO SUPERVISADO  │  │    POR    │  │
│  │                     │  │                  │  │ REFUERZO  │  │
│  │ Aprender de         │  │ Encontrar        │  │           │  │
│  │ ejemplos etiquetados│  │ estructura en    │  │ Aprender  │  │
│  │                     │  │ datos sin        │  │ por       │  │
│  │ Ej: clasificar      │  │ etiquetar        │  │ prueba y  │  │
│  │ emails como spam    │  │                  │  │ error     │  │
│  │ o no-spam           │  │ Ej: agrupar      │  │           │  │
│  │                     │  │ clientes por     │  │ Ej: un    │  │
│  │                     │  │ comportamiento   │  │ agente    │  │
│  │                     │  │                  │  │ jugando   │  │
│  │                     │  │                  │  │ ajedrez   │  │
│  └─────────────────────┘  └──────────────────┘  └───────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

### ¿Por qué el axr es diferente?

| Característica | Supervisado | No supervisado | **Por refuerzo** |
|---|---|---|---|
| ¿Necesita datos etiquetados? | ✅ Sí | ❌ No | ❌ No |
| ¿Aprende de un maestro? | ✅ Sí | ❌ No | ❌ No |
| ¿Aprende interactuando? | ❌ No | ❌ No | ✅ **Sí** |
| ¿Tiene en cuenta el futuro? | Raramente | No | ✅ **Sí** |
| ¿Recibe recompensas? | ❌ No | ❌ No | ✅ **Sí** |

💡 **Punto clave:** En el axr, nadie le dice al agente qué hacer. El agente tiene que **descubrir** por sí mismo qué acciones llevan a buenas recompensas, interactuando directamente con su entorno.

---
## 1. El aprendizaje por refuerzo

El aprendizaje por refuerzo explora una aproximación computacional al aprendizaje por interacción. De la misma manera que cuando aprendemos a conducir estamos atentos a cómo reacciona nuestro entorno a nuestras acciones y buscamos maneras de influenciarlo a través de nuestro comportamiento, el aprendizaje por refuerzo estudia cómo agentes computacionales pueden desarrollar comportamientos inteligentes a través de este tipo de interacción.

---
### 📖 La analogía del conductor — explicación extendida

Imagina que estás aprendiendo a conducir sin ningún instructor. Lo único que tienes es:
- Un **volante, frenos y acelerador** (tus *acciones*).
- La **vista de la carretera** y los demás vehículos (tu *percepción del entorno*).
- Una sensación de bienestar o peligro según lo que pasa (tu *recompensa*).

Nadie te dice "gira el volante exactamente 15 grados a la derecha ahora". Tú pruebas, observas qué pasa, y poco a poco aprendes qué funciona. Eso es exactamente el axr:

```
                    ┌─────────────────────────────────┐
                    │                                 │
          acción    │                                 │   observación
         ────────►  │         E N T O R N O           │  ──────────►
                    │                                 │   recompensa
   A G E N T E      │    (carretera, tablero de       │  ──────────►
   (el conductor)   │     juego, robot, etc.)         │
                    │                                 │
                    └─────────────────────────────────┘
```

El **agente** observa el entorno, toma una acción, recibe una recompensa (o penalización), y usa esa experiencia para mejorar sus decisiones futuras. Este ciclo se repite miles o millones de veces.

### 📖 Definición formal del axr

El **axr**, de ahora en adelante, consiste en aprender qué hacer (cómo relacionar situaciones y acciones) con el objetivo de maximizar una recompensa numérica. En ningún momento especificamos qué acciones debe tomar un agente, sino que le dejamos descubrir cuáles son las que le darán una mayor recompensa. En la mayoría de situaciones, una acción no solo afectará a la recompensa inmediata, sino que tendrá un efecto en todas las situaciones futuras. Estas dos propiedades, **búsqueda por prueba y error** y **futuras recompensas**, son las más importantes del axr.

Para formalizar el problema del axr utilizamos ideas del campo de la teoría de sistemas dinámicos, en concreto el **control óptimo de procesos de Markov incompletos**. La idea básica consiste en aprender los aspectos más importantes sobre el problema real al que nuestro agente se enfrenta a través de la interacción con el entorno para conseguir su objetivo. El agente tiene que ser capaz de percibir su entorno y de llevar a cabo acciones que afecten a su estado. También necesita uno o varios objetivos relacionados con el estado del entorno. Un **proceso de decisión de Markov (PDM)** incluye estos tres aspectos: percepción, acción y objetivo. Cualquier método que sea capaz de resolver este tipo de problemas se considera como un método de axr.

---
### 📖 ¿Qué es un "Proceso de Decisión de Markov"?

El nombre suena intimidante, pero la idea es sencilla. Un PDM es simplemente una forma matemática de describir el ciclo agente-entorno con cuatro ingredientes:

| Símbolo | Nombre | ¿Qué es? | Ejemplo en el tres en raya |
|---|---|---|---|
| $S$ | **Estado** | Descripción de la situación actual | La configuración del tablero |
| $A$ | **Acción** | Lo que el agente puede hacer | Poner una X en la celda (1,2) |
| $R$ | **Recompensa** | Señal numérica que recibe | +1 si gana, 0 si empata, -1 si pierde |
| $T$ | **Transición** | Cómo cambia el estado tras la acción | El tablero con la X añadida |

La palabra "Markov" significa que el **estado actual contiene toda la información relevante** — no necesitamos recordar toda la historia pasada para tomar buenas decisiones. Esto es una simplificación muy útil.

---
### 📖 El balance entre exploración y explotación

El axr está considerado como un paradigma del *machine learning* diferente al aprendizaje supervisado y no supervisado. Se diferencia del aprendizaje supervisado en que no siempre será posible obtener ejemplos del comportamiento deseado para nuestro agente en cualquier tipo de situación en la que se pueda encontrar, por lo que deberá ser capaz de aprender de su propia experiencia. Por otro lado, se diferencia del aprendizaje no supervisado ya que éste no es capaz por sí mismo de resolver el problema de maximización de la recompensa.

El principal problema al que nos enfrentamos en el axr es el balance entre **exploración** y **explotación**. Para obtener la máxima recompensa, un agente podría escoger aquellas acciones que ya conoce y que le dieron buenos resultados. Sin embargo, el hecho de explorar nuevas acciones podría, eventualmente, dar mucho mejor resultado. Este problema sigue sin estar resuelto.

---
### 📖 El dilema del restaurante — entendiendo exploración vs explotación

Imagina que estás en una ciudad nueva y tienes hambre. Conoces un restaurante que está "bien" (lo visitaste ayer). Tienes dos opciones:

- 🍽️ **Explotación**: ir al restaurante conocido. Recompensa **segura** pero posiblemente no la mejor.
- 🗺️ **Exploración**: probar un restaurante nuevo. Podría ser **horrible** o podría ser **increíble**.

Si siempre explotas, nunca descubres mejores opciones. Si siempre exploras, nunca disfrutas de lo que ya sabe que es bueno.

En el axr, este dilema aparece en cada paso. La solución más simple (que usaremos aquí) es la estrategia **ε-greedy** (épsilon-greedy):

```
Con probabilidad ε  → EXPLORAR  (acción aleatoria)
Con probabilidad 1-ε → EXPLOTAR (mejor acción conocida)
```

Si `ε = 0.5`, el agente explora el 50% de las veces. Si `ε = 0.1`, solo el 10%.

💡 **Punto clave:** Un `ε` alto al principio del entrenamiento es bueno — el agente no sabe nada aún, necesita explorar mucho. Con el tiempo, conviene reducir `ε` para que el agente explote más su conocimiento adquirido.

---
## 2. Ejemplos de aplicaciones del axr

Algunos ejemplos interesantes de aplicaciones del axr son:

- **Agentes que son capaces de jugar a juegos**: ajedrez, go, atari, starcraft ...
- **Sistemas de control adaptativo en entornos industriales**: refinerías, fábricas, cadenas de montaje ...
- **Robótica**
- **Conducción autónoma**: el coche recibe información de su entorno a través de sus cámaras y sensores y ejecuta comandos para acelerar, frenar, girar el volante ...

Como vemos, todos los ejemplos comparten la existencia de un agente en constante interacción con su entorno para lograr un objetivo (a pesar de la posible incertidumbre). Tomar una decisión puede afectar a las acciones y oportunidades futuras, por lo que la elección de una acción requiere capacidad de planificación y predicción. Además, gracias a su interacción con el entorno, un agente puede adaptarse y aprender constantemente, ajustándose si es necesario para mejorar. De todas las formas de inteligencia computacional, el axr es el que más se asemeja a la forma en la que personas y animales actuamos.

---
### 📖 ¿Por qué el axr es tan poderoso en juegos?

Los juegos son el campo de pruebas perfecto para el axr porque:

1. Las **reglas están bien definidas** → fácil de simular el entorno.
2. La **recompensa es clara** → ganar = +1, perder = -1.
3. Se pueden jugar **millones de partidas** en segundos → mucho entrenamiento.
4. Existe un **estándar de comparación** → el rendimiento humano.

El famoso **AlphaGo** (DeepMind, 2016) aprendió a jugar al Go mejor que cualquier humano usando axr. El Go tiene más posiciones posibles que átomos en el universo observable, por lo que es imposible programar reglas explícitas.

---
## 3. Elementos del axr

Además del agente y del entorno, existen cuatro subelementos esenciales en un sistema de axr:

- **Política**: define el comportamiento del agente en cada momento. Relaciona el estado que el agente percibe de su entorno con todas las posibles acciones que puede tomar. La política define completamente el comportamiento de un agente.
- **Recompensa**: define el objetivo del problema, y es un valor numérico que en cada momento el entorno envía al agente, el cual tiene el único objetivo de maximizarlo.
- **Función de valor**: mientras que la recompensa indica la calidad de un estado de manera inmediata, la función de valor indica la calidad a largo plazo. El valor de un estado es la cantidad total de recompensa que un agente espera acumular en el futuro empezando en ese mismo estado.
- **Modelo** del entorno: imita el comportamiento del entorno y sirve para planificar acciones considerando estados futuros que todavía no se han experimentado.

---
### 📖 Los cuatro elementos explicados con el tres en raya

Veamos cada elemento en el contexto del juego que vamos a programar:

**1. Política (π)** — *¿Qué hago en cada situación?*
La política es como el "instinto" del agente. Si el tablero tiene esta configuración, ¿dónde pongo mi X? La política podría ser:
- *Simple*: "siempre pon en la esquina superior izquierda si está libre"
- *Aprendida*: "consulta la función de valor y elige la celda que lleva al estado más prometedor"

**2. Recompensa (R)** — *¿Qué tan bien lo hice ahora mismo?*
- Gané la partida → **R = 1**
- Perdí la partida → **R = 0**
- Empate → **R = 0.5**
- Movimiento intermedio → no hay recompensa aún (esperamos al final)

**3. Función de valor V(S)** — *¿Qué tan prometedor es este estado a largo plazo?*
Es la idea más importante. `V(S)` responde a: "desde esta posición del tablero, ¿cuál es mi probabilidad de ganar?"

```
Tablero A:  X _ _     V(A) ≈ 0.5  (partida recién empezada, incierta)
            _ _ _
            _ _ _

Tablero B:  X _ X     V(B) ≈ 0.8  (X está a punto de ganar en la fila 1)
            _ O _
            _ _ O

Tablero C:  X X X     V(C) = 1.0  (¡X ganó! Estado terminal ganador)
            O _ O
            _ O _
```

**4. Modelo del entorno** — *¿Puedo predecir qué pasará si hago X?*
En este cuadernillo **no usamos un modelo explícito**. El agente aprende directamente de la experiencia (método sin modelo = *model-free*). Esto es más simple y más general.

---
### 💡 Recompensa vs. Función de valor — la diferencia clave

Piensa en esto:
- **Recompensa** = placer o dolor *inmediato*. Comer azúcar da placer ahora.
- **Función de valor** = bienestar *a largo plazo*. El azúcar en exceso reduce la salud futura.

Un agente inteligente debe **favorecer acciones de mayor valor**, no solo de mayor recompensa inmediata. Por eso la función de valor es más útil que la recompensa para guiar las decisiones.

---
## 4. Ejemplo de aplicación: el tres en raya

Con el objetivo de ilustrar la idea general del axr vamos a considerar un ejemplo en detalle: el juego del **tres en raya** (también conocido como *tic-tac-toe*). En este juego, dos jugadores se turnan para dibujar una X o una O en un tablero con 3×3 posiciones. El primer jugador en conseguir dibujar tres figuras en una línea horizontal, vertical o diagonal, gana. Nuestro objetivo es conseguir un agente que sea capaz de ganar siempre a este juego.

![](https://camo.githubusercontent.com/9b5f16d0451a8b6faf1ec45a6afbe22f55bbe1f9/68747470733a2f2f7468756d62732e6766796361742e636f6d2f506f697365644772697070696e67466f782d736d616c6c2e676966)

---
### 📖 ¿Por qué el tres en raya es un buen ejemplo introductorio?

El tres en raya tiene tres propiedades que lo hacen ideal para aprender axr:

1. **Espacio de estados manejable**: hay aproximadamente 5 478 posiciones posibles del tablero. Una computadora puede explorarlas todas.
2. **Reglas simples**: es fácil programar qué movimientos son válidos y cuándo termina el juego.
3. **Resultado binario**: se gana, se pierde o se empata. La recompensa está clara.

A pesar de su simplicidad, el proceso de aprendizaje que vamos a implementar es **exactamente el mismo** que se usa en sistemas más complejos como AlphaGo o los agentes de Atari.

---
### 📖 Cómo representar el estado del tablero en código

Un tablero de tres en raya es una cuadrícula 3×3. Necesitamos una forma de representarlo en la computadora. Usaremos una matriz de números:

- `0` → celda vacía
- `1` → X (jugador 1)
- `-1` → O (jugador 2)

¿Por qué `1` y `-1` en lugar de `1` y `2`? Es una elección inteligente que simplifica las comprobaciones matemáticas. Por ejemplo, si una fila suma `3`, las tres celdas deben ser `1` (tres Xs). Si suma `-3`, son tres Os. Si suma `0` con tres fichas, hay una mezcla.

```python
# Ejemplo visual de un tablero y su representación numérica:
#
#  X | O | _        [[  1, -1,  0],
#  _ | X | _   →     [  0,  1,  0],
#  O | _ | X         [ -1,  0,  1]]
#
# Convertido a vector de 9 elementos:
#  [ 1. -1.  0.  0.  1.  0. -1.  0.  1.]
# Este vector es la "clave" en el diccionario de la función de valor.
```

Para guardar el estado en la función de valor (un diccionario), convertiremos la matriz de 3×3 en un **string** de 9 números. Así podemos usar el estado como clave del diccionario.

---
### 📖 La función de valor y su inicialización

Para resolver este juego con axr, en primer lugar definimos una tabla de números, uno por cada posible estado del juego. Cada número representará la probabilidad de ganar el juego desde ese estado, el *valor* del estado. Así pues, la tabla sería la *función de valor*. Un estado $A$ es considerado mejor que un estado $B$ si el valor estimado de la probabilidad de ganar el juego desde $A$ es mayor que desde $B$.

Si jugásemos con las Xs, todos los estados con tres X en raya tendrían un valor de 1 (hemos ganado). De la misma manera, cualquier estado con tres Os en raya tendría un valor de 0 (hemos perdido). Para la inicialización de la tabla, podemos establecer el resto de valores en **0.5** (50% de posibilidades de ganar).

Nuestro agente jugará muchas partidas contra un oponente (que puede ser otro agente). En cada turno evaluará los estados que resultarían de cada posible movimiento y elegirá aquel con mayor *valor*. Ocasionalmente, elegirá una acción aleatoria para explorar.

---
### 📖 La fórmula de actualización TD (Diferencia Temporal)

Mientras el agente juega, tiene que actualizar la función de valor para que sus estimaciones mejoren. Para ello, después de cada movimiento, cambiaremos el valor del estado del que venimos para que se acerque al valor del estado actual:

$$V(S_t) \leftarrow V(S_t) + \alpha [V(S_{t+1}) - V(S_t)]$$

Donde:
- $S_t$ es el estado **anterior** (antes del movimiento).
- $S_{t+1}$ es el estado **nuevo** (después del movimiento).
- $V(S_t)$ es el valor estimado actual del estado $S_t$.
- $\alpha$ es el **learning rate** (tasa de aprendizaje): controla cuánto cambiamos el valor en cada paso.

---
### 📖 Entendiendo la fórmula TD paso a paso

La fórmula parece intimidante, pero su lógica es muy intuitiva. Vamos a desglosarla:

**El término `[V(S_{t+1}) - V(S_t)]` es el "error de predicción"**

- Si $V(S_{t+1}) > V(S_t)$: el estado nuevo es mejor de lo que esperábamos. El error es **positivo**. Deberíamos subir el valor de $S_t$.
- Si $V(S_{t+1}) < V(S_t)$: el estado nuevo es peor. El error es **negativo**. Deberíamos bajar el valor de $S_t$.
- Si $V(S_{t+1}) = V(S_t)$: nuestra estimación era perfecta. No cambia nada.

**El término $\alpha$ controla la velocidad de aprendizaje:**
- $\alpha = 1.0$: aprendizaje máximo ("creo completamente en lo que acabo de ver").
- $\alpha = 0.1$: aprendizaje lento ("actualizo apenas un poco").
- Típicamente usamos $\alpha = 0.5$: un balance razonable.

```
Ejemplo numérico:

  V(S_t)   = 0.5  (estado actual, incertidumbre)
  V(S_t+1) = 0.8  (el estado siguiente parece muy bueno)
  α        = 0.5

  Error = 0.8 - 0.5 = 0.3  (¡el estado siguiente es mejor de lo esperado!)

  Actualización:
  V(S_t) ← 0.5 + 0.5 × 0.3
  V(S_t) ← 0.5 + 0.15
  V(S_t) ← 0.65  ✓ (subió, como esperábamos)
```

**¿Por qué se llama "Diferencia Temporal"?** Porque usamos la *diferencia* entre valores en dos *momentos temporales* consecutivos ($t$ y $t+1$) para aprender. No esperamos al final de la partida para actualizar — actualizamos en cada paso.

💡 **Punto clave:** La actualización TD es el corazón del axr. Con el tiempo, los valores "correctos" se propagan hacia atrás: los estados terminales (ganar/perder) tienen valores fijos, y los estados anteriores aprenden de ellos.

---
## 5. Implementación: la clase `Board` (el tablero)

Ahora pasamos al código. Vamos a implementar tres clases que en conjunto forman el sistema de axr:

```
┌──────────┐   usa   ┌──────────┐   usa   ┌──────────┐
│  Board   │ ◄────── │   Game   │ ──────► │  Agent   │
│(tablero) │         │ (juego)  │         │ (agente) │
└──────────┘         └──────────┘         └──────────┘
 Representa          Orquesta el           Aprende y
 el estado del       entrenamiento         toma
 juego               (selfplay)            decisiones
```

Comenzamos con `Board`. Su responsabilidad es mantener el estado del tablero, saber qué movimientos son válidos, y detectar cuándo termina el juego.

In [1]:
import numpy as np


class Board():
    def __init__(self):
        # Creamos el tablero como una matriz 3x3 llena de ceros.
        # 0 = celda vacía, 1 = X (jugador 1), -1 = O (jugador 2)
        self.state = np.zeros((3,3))

    def valid_moves(self):
        # Devuelve una lista de tuplas (fila, columna) para las celdas vacías.
        # La condición `self.state[i, j] == 0` filtra las posiciones libres.
        # Ejemplo de retorno: [(0,0), (0,2), (1,1), (2,0)] si esas celdas están libres.
        return [(i, j) for j in range(3) for i in range(3) if self.state[i, j] == 0]

    def update(self, symbol, row, col):
        # Coloca el símbolo del jugador (1 o -1) en la celda (row, col).
        # Primero comprueba que la celda esté vacía (== 0).
        if self.state[row, col] == 0:
            self.state[row, col] = symbol
        else:
            # Si la celda ya está ocupada, lanzamos un error.
            raise ValueError ("movimiento ilegal !")

    def is_game_over(self):
        # Comprueba si el juego ha terminado y devuelve:
        #   1  → ganó el jugador 1 (Xs)
        #  -1  → ganó el jugador 2 (Os)
        #   0  → empate (tablero lleno sin ganador)
        #  None → el juego continúa

        # --- Comprobar filas y columnas ---
        # .sum(axis=0) suma cada COLUMNA → vector de 3 valores
        # .sum(axis=1) suma cada FILA    → vector de 3 valores
        # Si alguna suma == 3, las tres celdas son +1 → ganó jugador 1
        # Si alguna suma == -3, las tres celdas son -1 → ganó jugador 2
        if (self.state.sum(axis=0) == 3).sum() >= 1 or (self.state.sum(axis=1) == 3).sum() >= 1:
            return 1
        if (self.state.sum(axis=0) == -3).sum() >= 1 or (self.state.sum(axis=1) == -3).sum() >= 1:
            return -1

        # --- Comprobar diagonales ---
        # Diagonal principal: (0,0), (1,1), (2,2)
        # Diagonal secundaria: (0,2), (1,1), (2,0)
        diag_sums = [
            sum([self.state[i, i] for i in range(3)]),       # diagonal principal
            sum([self.state[i, 3 - i - 1] for i in range(3)]),  # diagonal secundaria
        ]
        if diag_sums[0] == 3 or diag_sums[1] == 3:
            return 1
        if diag_sums[0] == -3 or diag_sums[1] == -3:
            return -1

        # --- Comprobar empate ---
        # Si no hay movimientos válidos y nadie ganó → empate
        if len(self.valid_moves()) == 0:
            return 0

        # --- Juego continúa ---
        return None

    def reset(self):
        # Reinicia el tablero a su estado inicial (todo ceros)
        # Se llama al comienzo de cada partida nueva.
        self.state = np.zeros((3,3))

### 🔍 Análisis de la clase `Board`

**`__init__`**: crea el tablero inicial. `np.zeros((3,3))` genera una matriz 3×3 de ceros usando NumPy.

**`valid_moves`**: usa una **comprensión de listas** para filtrar celdas vacías. Devuelve coordenadas `(fila, columna)`. Esta función es crucial porque el agente solo puede elegir entre movimientos válidos.

**`is_game_over`**: detecta el fin del juego con operaciones matriciales:
- `sum(axis=0)` suma las 3 columnas por separado.
- `sum(axis=1)` suma las 3 filas por separado.
- Si alguna suma es 3 o -3, hay una línea completa.
- Las diagonales se comprueban manualmente.

**`reset`**: simplemente sobrescribe el estado con ceros. Se usa entre partidas.

In [2]:
# ── Demostración del Board ─────────────────────────────────────────────────
# Vamos a simular manualmente unos pocos movimientos para ver cómo funciona.

tablero = Board()
print("Tablero inicial (todos ceros = vacío):")
print(tablero.state)
print("Movimientos válidos iniciales:", tablero.valid_moves())
print()

# Jugador 1 (X, símbolo=1) pone en el centro (1,1)
tablero.update(symbol=1, row=1, col=1)
# Jugador 2 (O, símbolo=-1) pone en la esquina superior izquierda (0,0)
tablero.update(symbol=-1, row=0, col=0)
# Jugador 1 pone en (0,2)
tablero.update(symbol=1, row=0, col=2)

print("Tablero tras 3 movimientos:")
print(tablero.state)
print()
print("¿Terminó el juego?", tablero.is_game_over())  # None = sigue jugando
print("Movimientos válidos restantes:", tablero.valid_moves())

Tablero inicial (todos ceros = vacío):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Movimientos válidos iniciales: [(0, 0), (1, 0), (2, 0), (0, 1), (1, 1), (2, 1), (0, 2), (1, 2), (2, 2)]

Tablero tras 3 movimientos:
[[-1.  0.  1.]
 [ 0.  1.  0.]
 [ 0.  0.  0.]]

¿Terminó el juego? None
Movimientos válidos restantes: [(1, 0), (2, 0), (0, 1), (2, 1), (1, 2), (2, 2)]


---
## 6. Implementación: la clase `Game` (el juego)

La clase `Game` es la que **orquesta el entrenamiento**. Su método principal, `selfplay`, hace que dos agentes jueguen miles de partidas entre sí. Así es como el aprendizaje ocurre: a través de la repetición.

### 📖 ¿Por qué "selfplay" (auto-juego)?

En lugar de entrenar contra un oponente fijo (por ejemplo, un bot con reglas escritas a mano), hacemos que **dos agentes aprendices se enfrenten entre sí**. Esto tiene ventajas importantes:

1. **Ambos mejoran simultáneamente** — el oponente siempre presenta un reto al nivel actual del agente.
2. **No necesitamos un "maestro" externo** — el sistema se mejora a sí mismo.
3. **Escala bien** — cuantas más partidas, mayor aprendizaje.

Esta técnica de selfplay es la misma que usó AlphaZero para dominar el ajedrez y el Go.

In [3]:
from tqdm import tqdm  # librería para mostrar barras de progreso


class Game():
    def __init__(self, player1, player2):
        # Asignamos símbolos a los jugadores.
        # player1 juega con +1 (Xs), player2 con -1 (Os).
        # Estos valores coinciden con la codificación del tablero.
        player1.symbol = 1
        player2.symbol = -1
        self.players = [player1, player2]  # lista de los dos jugadores
        self.board = Board()               # el tablero compartido

    def selfplay(self, rounds=100):
        wins = [0, 0]  # contador de victorias: wins[0] = player1, wins[1] = player2

        # Iteramos por `rounds` partidas.
        # tqdm() envuelve el rango para mostrar la barra de progreso.
        for i in tqdm(range(1, rounds + 1)):

            # --- Preparar nueva partida ---
            self.board.reset()         # tablero en blanco
            for player in self.players:
                player.reset()         # limpiar el historial de posiciones de cada agente

            game_over = False

            # --- Bucle de la partida ---
            while not game_over:
                # Los dos jugadores se turnan dentro del mismo bucle `while`.
                for player in self.players:

                    # 1. El jugador actual decide su movimiento.
                    action = player.move(self.board)
                    # action es una tupla (fila, columna)

                    # 2. Aplicamos el movimiento al tablero.
                    self.board.update(player.symbol, action[0], action[1])

                    # 3. AMBOS jugadores registran el nuevo estado del tablero.
                    # Esto es importante: los dos aprenden de cada movimiento.
                    for player in self.players:
                        player.update(self.board)

                    # 4. Comprobamos si la partida terminó.
                    if self.board.is_game_over() is not None:
                        game_over = True
                        break  # salir del bucle `for` de jugadores

            # --- Fin de la partida: asignar recompensas ---
            self.reward()

            # Actualizar contadores de victorias
            for ix, player in enumerate(self.players):
                if self.board.is_game_over() == player.symbol:
                    wins[ix] += 1

        return wins

    def reward(self):
        # Determina el resultado final y llama a reward() en cada agente.
        winner = self.board.is_game_over()

        if winner == 0:  # empate
            for player in self.players:
                player.reward(0.5)  # ambos reciben 0.5
        else:
            # Un jugador ganó. Damos 1 al ganador y 0 al perdedor.
            for player in self.players:
                if winner == player.symbol:
                    player.reward(1)   # ganó
                else:
                    player.reward(0)   # perdió

### 🔍 Análisis de la clase `Game`

**¿Por qué reseteamos al inicio de cada partida?**
- `self.board.reset()`: el tablero debe estar en blanco para cada nueva partida.
- `player.reset()`: cada agente guarda el historial de estados de la partida actual (para actualizar la función de valor al final). Hay que borrarlo entre partidas.

**¿Por qué ambos jugadores registran el estado tras cada movimiento?**
```python
for player in self.players:
    player.update(self.board)
```
Tanto el jugador que acaba de mover como el que espera necesitan saber cómo evolucionó el tablero. Así, al final de la partida, cada uno tiene el historial completo de estados para actualizar su función de valor.

**¿Por qué 0, 0.5 y 1 como recompensas?**
Usamos estos valores porque la función de valor también está inicializada en 0.5 y la fórmula TD trabaja en el rango [0, 1]. Podríamos usar otras escalas (como -1, 0, +1), pero esta elección simplifica el código.

### 📖 Flujo completo de una partida

```
INICIO DE PARTIDA
    │
    ▼
board.reset() + player.reset() para ambos
    │
    ▼
┌─────────────────────────────────────────┐
│  TURNO del Jugador 1                    │
│    action = player1.move(board)         │  ← decide dónde poner
│    board.update(+1, row, col)           │  ← actualiza tablero
│    player1.update(board)               │  ← guarda estado en historial
│    player2.update(board)               │  ← también guarda (ambos aprenden)
│    ¿terminó? → sí → FIN / no → sigue  │
│                                         │
│  TURNO del Jugador 2 (igual)            │
└─────────────────────────────────────────┘
    │
    ▼
FIN DE PARTIDA
    │
    ▼
game.reward()  →  player1.reward(1 o 0 o 0.5)
               →  player2.reward(1 o 0 o 0.5)
    │
    ▼
Ambos actualizan su función de valor con la fórmula TD
```

---
## 7. Implementación: la clase `Agent` (el corazón del axr)

Esta es la clase más importante. Aquí vive toda la lógica del aprendizaje por refuerzo: la función de valor, la exploración, y la actualización TD.

### 📖 Atributos del agente — antes de leer el código

| Atributo | Tipo | ¿Para qué sirve? |
|---|---|---|
| `value_function` | `dict` | La "memoria" del agente. Mapea cada estado (string) a su valor (0.0 – 1.0). |
| `alpha` | `float` | La tasa de aprendizaje (ej: 0.5). Cuánto pesa cada nueva experiencia. |
| `positions` | `list` | El historial de estados de la partida actual. Se usa para la actualización TD. |
| `prob_exp` | `float` | La probabilidad de exploración ε. Ej: 0.5 = explora el 50% de las veces. |
| `symbol` | `int` | Asignado por `Game`: +1 para X, -1 para O. |

La `value_function` es un **diccionario de Python**. La clave es el estado del tablero convertido a string, y el valor es un número entre 0 y 1. Inicialmente está vacío — el agente no sabe nada.

```python
# Ejemplo de cómo se ve value_function después de algunas partidas:
value_function = {
    '[ 1.  0.  0.  0.  0.  0.  0.  0.  0.]': 0.52,  # X en esquina, prometedor
    '[ 1.  0.  0.  0. -1.  0.  0.  0.  1.]': 0.71,  # X en esquinas, O en centro
    '[ 1.  1.  1. -1. -1.  0.  0.  0.  0.]': 1.00,  # X ganó → valor máximo
    '[-1. -1. -1.  1.  1.  0.  0.  0.  0.]': 0.00,  # O ganó → valor mínimo
}
```

In [3]:
class Agent():
    def __init__(self, alpha=0.5, prob_exp=0.5):
        self.value_function = {}   # tabla estado→valor (empieza vacía)
        self.alpha = alpha          # tasa de aprendizaje (learning rate)
        self.positions = []        # historial de estados de la partida actual
        self.prob_exp = prob_exp   # probabilidad de exploración (ε-greedy)

    def reset(self):
        # Limpia el historial al inicio de cada nueva partida.
        # NO borramos la value_function — ¡ese es el aprendizaje acumulado!
        self.positions = []

    def move(self, board, explore=True):
        valid_moves = board.valid_moves()  # lista de celdas disponibles

        # --- EXPLORACIÓN: acción aleatoria ---
        # np.random.uniform(0,1) genera un número aleatorio entre 0 y 1.
        # Si ese número es menor que prob_exp → explorar.
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            ix = np.random.choice(len(valid_moves))  # índice aleatorio
            return valid_moves[ix]                    # devuelve (fila, col) aleatoria

        # --- EXPLOTACIÓN: elegir la mejor acción conocida ---
        max_value = -1000  # valor más bajo posible (cualquier estado real lo superará)
        for row, col in valid_moves:
            # Simulamos el tablero si colocásemos nuestra ficha en (row, col)
            next_board = board.state.copy()   # copia del tablero actual
            next_board[row, col] = self.symbol  # colocamos nuestra ficha

            # Convertimos el tablero hipotético a string (clave del diccionario)
            next_state = str(next_board.reshape(3*3))

            # Buscamos el valor de este estado en la tabla.
            # Si no lo conocemos aún, asumimos 0 (desconocido = no prometedor).
            value = 0 if self.value_function.get(next_state) is None else self.value_function.get(next_state)

            # Guardamos el movimiento si su valor es el mejor hasta ahora
            if value >= max_value:
                max_value = value
                best_row, best_col = row, col

        return best_row, best_col

    def update(self, board):
        # Guarda el estado actual del tablero en el historial.
        # Convertimos la matriz 3×3 a un vector de 9 elementos y luego a string.
        # Este string será la clave del diccionario value_function. 
        self.positions.append(str(board.state.reshape(3*3)))

    def reward(self, reward):
        # Al final de la partida, recorremos el historial EN ORDEN INVERSO
        # y actualizamos cada estado con la fórmula TD:
        #   V(S_t) ← V(S_t) + alpha * (recompensa_ajustada - V(S_t))
        #
        # ORDEN INVERSO: empezamos desde el último estado (más cercano al resultado)
        # y propagamos el aprendizaje hacia atrás en el tiempo.
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0  # inicializar si es la primera vez
            # Aplicar fórmula TD
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            # La recompensa se convierte en el valor del estado recién actualizado,
            # lo que propaga la información hacia estados más antiguos.
            reward = self.value_function[p]

### 🔍 Análisis profundo del método `reward` — la magia del TD

Este método es el más importante de todo el cuadernillo. Vamos a trazarlo con un ejemplo completo.

Supón que el agente acaba de ganar una partida (`reward = 1`). Su historial de posiciones fue:

```
positions = [estado_1, estado_2, estado_3, estado_4]  (en orden temporal)
           inicio                              fin (el agente ganó aquí)
```

La actualización en orden **inverso** funciona así:

```
reward = 1.0  (recompensa final: ganó)

Paso 1 - estado_4 (el último antes de ganar):
  V[estado_4] era 0 (desconocido)
  V[estado_4] = 0 + 0.5 × (1.0 - 0) = 0.50
  reward = 0.50  ← la recompensa ahora es el valor actualizado

Paso 2 - estado_3:
  V[estado_3] era 0 (desconocido)
  V[estado_3] = 0 + 0.5 × (0.50 - 0) = 0.25
  reward = 0.25

Paso 3 - estado_2:
  V[estado_2] era 0 (desconocido)
  V[estado_2] = 0 + 0.5 × (0.25 - 0) = 0.125
  reward = 0.125

Paso 4 - estado_1 (el estado inicial):
  V[estado_1] = 0 + 0.5 × (0.125 - 0) = 0.0625
```

**Observaciones importantes:**
1. Los estados **más cercanos a la victoria** obtienen valores **más altos** (0.50 > 0.25 > 0.125).
2. Con **más partidas**, estos valores se acumulan y mejoran. Después de ganar muchas veces desde ciertos estados, sus valores subirán cerca de 1.0.
3. Después de perder (reward = 0), los valores de los estados recorridos bajarán.

### ✏️ Reflexión: ¿por qué reversed?

Si actualizásemos en orden **normal** (de principio a fin), la señal de la victoria final tardaría muchas partidas en llegar a los estados del inicio. Yendo en orden inverso, la información fluye hacia atrás eficientemente en una sola pasada.

### 🔍 ¿Por qué el valor de estados desconocidos es 0 (y no 0.5)?

Nota que en `move()`, los estados desconocidos tienen valor 0, pero en `reward()` se inicializan a 0. La inicialización en 0 en `reward()` tiene sentido: si un estado nunca fue visitado antes, empezamos "pesimistas" (valor 0) y lo actualizamos hacia arriba si lleva a buenos resultados. Esto es diferente a la inicialización en 0.5 mencionada en la teoría, pero en la práctica funciona bien porque los estados son visitados rápidamente.

---
## 8. Entrenamiento

Ahora tenemos todas las piezas. Vamos a crear dos agentes y hacerlos jugar 300 000 partidas entre sí.

### 📖 ¿Por qué 300 000 partidas?

El espacio de estados del tres en raya tiene ~5 478 estados posibles. Para que el agente los explore todos con suficiente frecuencia y estime bien los valores, necesita muchas partidas. Con `prob_exp = 0.5`, la mitad de los movimientos son aleatorios, garantizando que se explorarán estados poco comunes.

**¿Qué está pasando durante el entrenamiento?**

En cada una de las 300 000 partidas:
1. Los agentes juegan una partida completa (~5-9 movimientos).
2. Uno gana (o empatan).
3. Ambos actualizan su función de valor.
4. Los valores mejoran poco a poco: los estados que llevan a victorias suben, los que llevan a derrotas bajan.
5. Al cabo de miles de partidas, los agentes empiezan a preferir posiciones más prometedoras.

Al final, `agent1.value_function` contendrá un diccionario con ~5 477 entradas, una por cada estado del tablero que se visitó.

In [5]:
# Creamos dos agentes con los hiperparámetros por defecto:
#   alpha = 0.5    → tasa de aprendizaje moderada
#   prob_exp = 0.5 → explora el 50% de las veces
agent1 = Agent(prob_exp=0.5)
agent2 = Agent()

# Creamos el juego con los dos agentes.
# Game asigna: agent1.symbol = 1, agent2.symbol = -1
game = Game(agent1, agent2)

# ¡A entrenar! Esto puede tardar unos minutos.
# tqdm mostrará el progreso y la velocidad (partidas/segundo).
game.selfplay(300000)

100%|██████████| 300000/300000 [17:38<00:00, 283.40it/s]


[171012, 78107]

### 📖 Interpretando el resultado del entrenamiento

`selfplay` devuelve una lista `[victorias_agent1, victorias_agent2]`. El número restante hasta 300 000 son empates.

El agente que va primero (agent1, las Xs) tiene **ventaja natural** en el tres en raya: con juego perfecto, puede forzar al menos un empate desde cualquier posición y ganar si el oponente comete errores. Por eso agent1 suele tener más victorias.

**Nota:** El resultado exacto variará en cada ejecución porque hay aleatoriedad en las partidas (exploración). Pero el patrón general (agent1 > agent2 en victorias) debería mantenerse.

---
## 9. Inspeccionando la función de valor aprendida

Después del entrenamiento, `agent1.value_function` es el "cerebro" del agente. Vamos a mirarlo.

### 📖 ¿Cómo leer la función de valor?

Cada fila de la tabla es un estado del tablero que el agente visitó, junto con el valor que aprendió para ese estado. El **valor** es la estimación del agente de su probabilidad de ganar desde esa posición:

- **Valor ≈ 1.0**: el agente casi seguro gana desde ese estado.
- **Valor ≈ 0.5**: estado incierto, difícil de predecir.
- **Valor ≈ 0.0**: el agente casi seguro pierde desde ese estado.

In [6]:
import pandas as pd

# Ordenamos los estados de mayor a menor valor para ver los mejores y peores.
# `items()` devuelve pares (clave, valor) del diccionario.
# `sorted(..., key=lambda kv: kv[1], reverse=True)` ordena por valor descendente.
funcion_de_valor = sorted(agent1.value_function.items(), key=lambda kv: kv[1], reverse=True)

# Creamos un DataFrame de pandas para visualizarlo mejor.
tabla = pd.DataFrame({
    'estado': [x[0] for x in funcion_de_valor],  # string del tablero
    'valor':  [x[1] for x in funcion_de_valor]    # probabilidad estimada de ganar
})

tabla

,estado,valor
0,[-1. 1. -1. 1. 1. 1. 0. 0. -1.],1.0
1,[ 0. -1. 1. 0. 1. -1. 1. -1. 1.],1.0
2,[ 1. -1. 1. 1. -1. -1. 1. 1. -1.],1.0
3,[ 0. -1. 1. 1. -1. -1. 1. 1. -1.],1.0
4,[ 1. 1. 1. 0. 1. -1. 0. -1. -1.],1.0
...,...,...
5472,[-1. 1. -1. 1. -1. 1. -1. 1. 0.],0.0
5473,[-1. 1. 0. 1. 0. -1. 1. 1. -1.],0.0
5474,[ 0. 1. -1. -1. 0. 1. -1. 1. 1.],0.0
5475,[-1. 1. 1. -1. 0. -1. 0. 1. 1.],0.0


In [7]:
# ── Decodificando un estado concreto ──────────────────────────────────────
# Los estados son strings que representan vectores de 9 números.
# Vamos a decodificar el estado con mayor valor para entender qué tablero representa.

def decodificar_estado(estado_str, valor):
    """Convierte el string del estado en una visualización del tablero."""
    # Parseamos el string para obtener los 9 números
    import re
    numeros = re.findall(r'[-]?\d+\.', estado_str)
    numeros = [float(n) for n in numeros]

    simbolos = {1.0: ' X ', -1.0: ' O ', 0.0: ' _ '}
    print(f"Valor: {valor:.4f}")
    print("┌───┬───┬───┐")
    for fila in range(3):
        celdas = [simbolos.get(numeros[fila*3 + col], ' ? ') for col in range(3)]
        print("│" + "│".join(celdas) + "│")
        if fila < 2:
            print("├───┼───┼───┤")
    print("└───┴───┴───┘")
    print()

# Mostrar los 5 estados con mayor valor (estados donde X casi seguro gana)
print("=== Top 5 estados más favorables para el agente (X) ===")
for estado, valor in funcion_de_valor[:5]:
    decodificar_estado(estado, valor)

# Mostrar los 5 estados con menor valor (estados donde X casi seguro pierde)
print("=== Top 5 estados menos favorables para el agente (X) ===")
for estado, valor in funcion_de_valor[-5:]:
    decodificar_estado(estado, valor)

print(f"Total de estados aprendidos: {len(agent1.value_function)}")

=== Top 5 estados más favorables para el agente (X) ===
Valor: 1.0000
┌───┬───┬───┐
│ O │ X │ O │
├───┼───┼───┤
│ X │ X │ X │
├───┼───┼───┤
│ _ │ _ │ O │
└───┴───┴───┘

Valor: 1.0000
┌───┬───┬───┐
│ _ │ O │ X │
├───┼───┼───┤
│ _ │ X │ O │
├───┼───┼───┤
│ X │ O │ X │
└───┴───┴───┘

Valor: 1.0000
┌───┬───┬───┐
│ X │ O │ X │
├───┼───┼───┤
│ X │ O │ O │
├───┼───┼───┤
│ X │ X │ O │
└───┴───┴───┘

Valor: 1.0000
┌───┬───┬───┐
│ _ │ O │ X │
├───┼───┼───┤
│ X │ O │ O │
├───┼───┼───┤
│ X │ X │ O │
└───┴───┴───┘

Valor: 1.0000
┌───┬───┬───┐
│ X │ X │ X │
├───┼───┼───┤
│ _ │ X │ O │
├───┼───┼───┤
│ _ │ O │ O │
└───┴───┴───┘

=== Top 5 estados menos favorables para el agente (X) ===
Valor: 0.0000
┌───┬───┬───┐
│ O │ X │ O │
├───┼───┼───┤
│ X │ O │ X │
├───┼───┼───┤
│ O │ X │ _ │
└───┴───┴───┘

Valor: 0.0000
┌───┬───┬───┐
│ O │ X │ _ │
├───┼───┼───┤
│ X │ _ │ O │
├───┼───┼───┤
│ X │ X │ O │
└───┴───┴───┘

Valor: 0.0000
┌───┬───┬───┐
│ _ │ X │ O │
├───┼───┼───┤
│ O │ _ │ X │
├───┼───┼───┤
│ O │ X │ X

### 📖 ¿Qué observamos en la función de valor?

Al inspeccionar los estados con valor `1.0`, veremos tableros donde **X tiene tres en raya** — estos son estados terminales ganadores, con valor perfectamente ajustado a 1.

Los estados con valor `0.0` corresponden a tableros donde **O tiene tres en raya** — el agente aprendió que desde ahí ya no puede ganar.

Los estados intermedios (valores entre 0.3 y 0.7) son posiciones donde el resultado aún no está decidido. Cuanto más cerca de 1, más ventajosa la posición para X.

✏️ **Reflexión:** El agente nunca recibió instrucciones explícitas sobre estrategia. Nadie le dijo "el centro es una posición fuerte" o "bloquea cuando el oponente tiene dos en raya". Aprendió estas cosas **por sí solo** a través de las recompensas de miles de partidas.

Esto ilustra el poder del axr: el agente descubre estrategias de forma autónoma.

---
## 10. Guardando el agente entrenado

El entrenamiento tomó minutos. No queremos repetirlo cada vez. Guardamos la función de valor en un archivo `.pickle` para reutilizarla.

### 📖 ¿Qué es pickle?

**Pickle** es un módulo de Python que **serializa** objetos Python — los convierte a bytes y los guarda en un archivo. Más tarde podemos cargar ese archivo y recuperar el objeto original exactamente igual.

Es como congelar el estado mental del agente en un archivo.

```
agent1.value_function  →  pickle.dump()  →  'agente.pickle' (archivo en disco)
                                              ↓
                                           pickle.load()  →  value_function cargada
```

In [10]:
import pickle

# Abrimos el archivo en modo escritura binaria ('wb' = write binary)
# y volcamos el diccionario value_function con pickle.dump().
# protocol=pickle.HIGHEST_PROTOCOL usa el formato más eficiente disponible.
with open('agente.pickle', 'wb') as handle:
    pickle.dump(agent1.value_function, handle, protocol=pickle.HIGHEST_PROTOCOL)

print("Agente guardado en 'agente.pickle'")
print(f"Tamaño de la función de valor: {len(agent1.value_function)} estados")

Agente guardado en 'agente.pickle'
Tamaño de la función de valor: 5477 estados


In [4]:
import pickle
# ── Demostración: cómo cargar el agente en una sesión futura ───────────────

# Abrimos el archivo en modo lectura binaria ('rb' = read binary)
with open('agente.pickle', 'rb') as handle:
    value_function_cargada = pickle.load(handle)

# Creamos un nuevo agente y le asignamos la función de valor cargada.
# prob_exp=0.0 → sin exploración: el agente usará solo lo que sabe.
agente_cargado = Agent(prob_exp=0.0)
agente_cargado.value_function = value_function_cargada

print(f"Agente cargado correctamente con {len(agente_cargado.value_function)} estados.")

Agente cargado correctamente con 5477 estados.


---
## 11. Reflexión sobre el ejemplo

Este ejemplo sirve para ilustrar algunas de las propiedades clave del axr. En primer lugar, **aprender a través de la interacción con el entorno** (en este caso el otro agente). En segundo lugar, tenemos un objetivo claro y el comportamiento correcto del agente requiere de **planificación y predicción** que tenga en cuenta los efectos futuros de sus acciones.

---
## Resumen

El aprendizaje por refuerzo es una aproximación computacional a la comprensión y automatización del aprendizaje por objetivos y toma de decisiones. En esta aproximación, un agente aprende a través de la interacción directa con su entorno sin necesidad de supervisión explícita. Utiliza **procesos de decisión de Markov** para definir la interacción entre el agente y su entorno en términos de estados, acciones y recompensas. Los conceptos de **valor y función de valor** son la clave de muchos métodos de axr ya que representan una manera eficiente de búsqueda en el espacio de políticas.

---
### 📖 Resumen expandido — lo que aprendimos en este cuadernillo

**Conceptos clave:**

| Concepto | Definición sencilla | En nuestro ejemplo |
|---|---|---|
| **Agente** | El que aprende y actúa | El jugador de tres en raya |
| **Entorno** | Con lo que el agente interactúa | El tablero y el oponente |
| **Estado (S)** | La situación actual | La configuración del tablero |
| **Acción (A)** | Lo que el agente puede hacer | Poner ficha en una celda |
| **Recompensa (R)** | Señal de qué tan bien lo hizo | +1 ganar, 0 perder, 0.5 empatar |
| **Función de valor V(S)** | Estimación de buenas perspectivas | P(ganar) desde ese tablero |
| **Política (π)** | Cómo decide el agente | Elegir la celda de mayor valor |
| **Exploración** | Probar cosas nuevas | Movimiento aleatorio con prob. ε |
| **Explotación** | Usar el conocimiento actual | Elegir la mejor celda conocida |
| **Actualización TD** | Cómo mejora la estimación | $V(S_t) \leftarrow V(S_t) + \alpha[V(S_{t+1})-V(S_t)]$ |
| **Selfplay** | Entrenamiento por auto-juego | Dos agentes jugando entre sí |

**El flujo completo del aprendizaje:**

```
  [Agente sin conocimiento] 
         ↓
  Juega partidas (exploit 50% + explore 50%)
         ↓
  Al final de cada partida: actualiza V(S) con fórmula TD
         ↓
  Los estados que llevan a ganar → V sube
  Los estados que llevan a perder → V baja
         ↓
  [Agente que prefiere posiciones ventajosas]
         ↓
  (300 000 partidas después)
         ↓
  [Agente que juega casi óptimo]
```

---
## Glosario de términos

| Término | Español | Definición |
|---|---|---|
| **RL / axr** | Aprendizaje por Refuerzo | Paradigma del ML donde un agente aprende interactuando con su entorno |
| **Agent** | Agente | El "jugador" que aprende y toma decisiones |
| **Environment** | Entorno | El mundo con el que interactúa el agente |
| **State** | Estado | Descripción de la situación actual |
| **Action** | Acción | Lo que el agente puede hacer en un estado |
| **Reward** | Recompensa | Señal numérica que indica el éxito inmediato |
| **Value function** | Función de valor | Estimación del éxito futuro desde un estado |
| **Policy** | Política | Regla que dice qué acción tomar en cada estado |
| **Alpha (α)** | Tasa de aprendizaje | Cuánto peso se da a la nueva experiencia |
| **Epsilon (ε)** | Tasa de exploración | Probabilidad de elegir una acción aleatoria |
| **TD** | Diferencia Temporal | Técnica de actualización que usa diferencias entre valores consecutivos |
| **Selfplay** | Auto-juego | Entrenar haciendo que el agente juegue contra sí mismo |
| **MDP** | Proceso de Decisión de Markov | Modelo matemático del ciclo agente-entorno |
| **Pickle** | — | Módulo Python para guardar objetos en archivos |

---
## Próximos pasos

Si este cuadernillo te resultó comprensible, te recomendamos explorar:

1. **Modificar los hiperparámetros**: ¿qué pasa si `alpha=0.1` o `prob_exp=0.9`? ¿Cómo afecta al aprendizaje?
2. **Jugar contra el agente**: usa `agente_cargado` para crear una interfaz interactiva donde tú juegas contra el agente.
3. **Reducir la exploración progresivamente**: empieza con `prob_exp=0.8` y ve bajándola a medida que avanza el entrenamiento.
4. **Q-Learning**: una extensión del TD donde el agente aprende la función Q(estado, acción) en lugar de V(estado).
5. **Deep RL**: sustituir la tabla de valores por una red neuronal para manejar espacios de estados mucho más grandes.

El axr es un campo enorme con aplicaciones fascinantes. Este cuadernillo es apenas la punta del iceberg, pero los conceptos fundamentales — estado, recompensa, función de valor, exploración/explotación y actualización TD — son los mismos en todos los sistemas más avanzados.